In [9]:
import pandas as pd
import numpy as np

df = pd.read_csv("./data/log_sensor_fix.csv")
# df = pd.read_csv("./data/sintetis/dataset_mentah.csv")
print(f"Data: {df.shape[0]} baris, {df.shape[1]} kolom")
df.head()

Data: 257 baris, 9 kolom


,Timestamp,soil_moisture,soil_temperature,air_temperature,air_humidity,nitrogen,fosfor,kalium,ec
0,2026-07-05T09:07:11Z,23.7,8.4,24.4,75.6,128,232,163,3829
1,2026-07-05T09:08:12Z,23.5,8.4,24.4,75.5,128,232,163,3831
2,2026-07-05T09:09:12Z,23.3,8.4,24.4,75.9,127,232,163,3833
3,2026-07-05T09:10:15Z,23.2,8.4,24.5,76.2,126,232,162,3835
4,2026-07-05T09:11:15Z,23.0,8.4,24.5,76.0,125,232,162,3836


# Cek data

In [10]:
df[["soil_moisture", "air_temperature", "air_humidity"]].describe().round(2)

,soil_moisture,air_temperature,air_humidity
count,257.00,257.00,257.00
mean,15.59,22.29,77.75
std,7.10,5.83,17.30
min,9.90,15.20,40.00
25%,11.00,18.00,71.90
50%,13.50,20.60,87.30
75%,18.00,24.50,89.90
max,74.60,34.30,93.20


## hapus data ga kepake

In [11]:
n_awal = len(df)

# --- Buang baris dengan soil_moisture == 0 (sensor mati) ---
df_bersih = df[df["soil_moisture"] != 0].reset_index(drop=True)

In [12]:
# Konversi EC dari uS/cm -> mS/cm (dS/m) HANYA jika masih dalam ribuan
if df["ec"].max() > 100:        
    df["ec"] = (df["ec"] / 1000).round(2)

# Fungsi labelling

In [ ]:
# acuan diambil dari petani 
#   - siram pagi  : soil_moisture ~11  (udara sejuk & lembap)
#   - siram jam 11: soil_moisture ~25  (udara panas 33C & kering 42%)

def label_irigasi(row):
    sm = row["soil_moisture"]
    at = row["air_temperature"]
    ah = row["air_humidity"]
    ec = row.get("ec", None)

    if sm == 0 or ec == 0:      
        return -1
    if sm > 45:                 
        return -1

    threshold_kering = 12
    threshold_basah = 20

    if at > 32 and ah < 55:     
        threshold_kering += 13  
        threshold_basah += 10   
    elif ah > 85:               
        threshold_kering -= 1
    
    if sm < threshold_kering:
        action = 1              
    else:
        action = 0              

    return action

print("Fungsi label_irigasi() siap.")


Fungsi label_irigasi() siap.


# Terapkan labelling

In [14]:
df["irrigation_action"] = df.apply(label_irigasi, axis=1)

df = df[df["irrigation_action"] != -1].reset_index(drop=True) 

print("Distribusi label:")
dist = df["irrigation_action"].value_counts().sort_index()
for val, count in dist.items():
    label = "Tidak siram" if val == 0 else "Siram"
    print(f"  {val} ({label}): {count} ({count/len(df)*100:.1f}%)")


Distribusi label:
  0 (Tidak siram): 161 (63.1%)
  1 (Siram): 94 (36.9%)


# Contoh

In [15]:
sample = df[["soil_moisture", "air_temperature",
             "air_humidity", "irrigation_action"]].head(20)
sample

,soil_moisture,air_temperature,air_humidity,irrigation_action
0,23.7,24.4,75.6,0
1,23.5,24.4,75.5,0
2,23.3,24.4,75.9,0
3,23.2,24.5,76.2,0
4,23.0,24.5,76.0,0
5,22.2,24.4,76.7,0
6,21.6,24.6,75.8,0
7,21.2,24.5,77.7,0
8,20.7,24.5,78.1,0
9,20.4,24.4,78.3,0


# Simpan

In [16]:
FEATURES = ["soil_moisture", "air_temperature", "air_humidity"]
cols = FEATURES + ["irrigation_action"]

df[cols].to_csv("data/dataset_irigasi.csv", index=False)
print("Tersimpan: data/dataset_irigasi.csv")
print(f"Total: {len(df)} baris, kolom: {cols}")


Tersimpan: data/dataset_irigasi.csv
Total: 255 baris, kolom: ['soil_moisture', 'air_temperature', 'air_humidity', 'irrigation_action']
